<a href="https://colab.research.google.com/github/jingwen320/skinmate_application/blob/feature%2Fmodel/skinmate_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install roboflow
from google.colab import drive
drive.mount('/content/drive')

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
import os

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 195.8/195.8 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 65.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 115.6 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.13.0.92
    Uninstalling opencv-python-headless-4.13.0.92:
      Successfully uninstalled opencv-python-headless-4.13.0.92
  Attempting uninstall: idna
    Found existing installation: idna 3.13
    Uninstalling idna-3.13:
      Successfully uninstalled idna-3.13
Mounted at /content/drive


In [2]:
from roboflow import Roboflow
rf = Roboflow(api_key="pzGHYwQsdfFMGP7mWvDF")
project = rf.workspace("fyp-symvg").project("skin-type-ng2uj")
version = project.version(1)
dataset = version.download("folder")

train_dir = '/content/skin-type-1/train'
val_dir = '/content/skin-type-1/valid'

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

val_datagen = ImageDataGenerator(rescale=1./255)

type_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

type_val_generator = val_datagen.flow_from_directory(
    val_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to skin-type-1 in folder:: 100%|██████████| 14198/14198 [00:03<00:00, 3840.85it/s]


Found 12408 images belonging to 4 classes.
Found 1181 images belonging to 4 classes.


In [3]:
base_model = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = True # Critical for 85%+ accuracy

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.3)(x)
predictions = Dense(4, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

# Define the 'checkpoint' that the error was complaining about
checkpoint = ModelCheckpoint(
    'skinmate_type_best_v2.keras',
    monitor='val_accuracy',
    save_best_only=True,
    mode='max',
    verbose=1
)

# Define the 'early_stop'
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

print("Callbacks are now defined! You can run the model.fit() cell again.")

Callbacks are now defined! You can run the model.fit() cell again.


In [4]:
import tensorflow as tf

path = '/content/drive/MyDrive/skinmate_type_best_v2.keras'
model = tf.keras.models.load_model(path)

optimizer = tf.keras.optimizers.Adam(learning_rate=3e-6)

model.compile(
    optimizer=optimizer,
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

checkpoint = tf.keras.callbacks.ModelCheckpoint(
    filepath='/content/drive/MyDrive/skinmate_type_best_v2.keras',
    monitor='val_accuracy',
    save_best_only=True,
    verbose=1
)

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_accuracy',
    patience=10,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_accuracy',
    factor=0.5,
    patience=4,
    verbose=1
)

print("✅ Model loaded. Ready to push for 85%!")

✅ Model loaded. Ready to push for 85%!


In [ ]:
print(type_generator.class_indices)

{'combination': 0, 'dry': 1, 'normal': 2, 'oily': 3}


In [ ]:
# This cell actually starts the training
history = model.fit(
    type_generator,
    steps_per_epoch=len(type_generator),
    validation_data=type_val_generator,
    validation_steps=len(type_val_generator),
    epochs=50,
    callbacks=[checkpoint, early_stop, reduce_lr] # reduce_lr is now included
)

Epoch 1/50
388/388 ━━━━━━━━━━━━━━━━━━━━ 0s 6s/step - accuracy: 0.9239 - loss: 0.2170
Epoch 1: val_accuracy improved from None to 0.81795, saving model to /content/drive/MyDrive/skinmate_type_best_v2.keras

Epoch 1: finished saving model to /content/drive/MyDrive/skinmate_type_best_v2.keras
388/388 ━━━━━━━━━━━━━━━━━━━━ 2619s 7s/step - accuracy: 0.9245 - loss: 0.2142 - val_accuracy: 0.8180 - val_loss: 0.5760 - learning_rate: 3.0000e-06
Epoch 2/50
388/388 ━━━━━━━━━━━━━━━━━━━━ 0s 6s/step - accuracy: 0.9247 - loss: 0.2192
Epoch 2: val_accuracy improved from 0.81795 to 0.82134, saving model to /content/drive/MyDrive/skinmate_type_best_v2.keras

Epoch 2: finished saving model to /content/drive/MyDrive/skinmate_type_best_v2.keras
388/388 ━━━━━━━━━━━━━━━━━━━━ 2445s 6s/step - accuracy: 0.9259 - loss: 0.2114 - val_accuracy: 0.8213 - val_loss: 0.5777 - learning_rate: 3.0000e-06
Epoch 3/50
388/388 ━━━━━━━━━━━━━━━━━━━━ 0s 7s/step - accuracy: 0.9253 - loss: 0.2128
Epoch 3: val_accuracy did not improv